### 1.2. Configuration

In [ ]:
# Run configuration
coco_subset_percent = 10
train_model = True

# Training hyperparameters
epochs = 30
batch_size = 8
learning_rate = 1e-4
image_size = 256

# Loss weights
content_weight = 1.0
style_weight = 100.0
tv_weight = 1e-4

# Create directories
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

logs_dir = Path('../logs')
logs_dir.mkdir(exist_ok=True)

styles_dir = Path('../data/styles')
styles_dir.mkdir(parents=True, exist_ok=True)

# Random seeds
np.random.seed(42)
tf.random.set_seed(42)

# GPU configuration
gpu_id = 0
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        if gpu_id is not None:
            tf.config.set_visible_devices(gpus[gpu_id], 'GPU')
            tf.config.experimental.set_memory_growth(gpus[gpu_id], True)
            print(f'Using GPU {gpu_id}: {gpus[gpu_id].name}')
    except RuntimeError as e:
        print(e)
else:
    print('No GPU available, using CPU')

## 2. Data preparation

We need two types of images:
- **Content images**: COCO dataset photos (what we want to stylize)
- **Style images**: Famous artworks (styles to transfer)

### 2.1. Load content images (COCO)

In [ ]:
# Load COCO dataset at higher resolution
print("Loading COCO dataset...")
(x_train, y_train), (x_test, y_test) = load_coco(
    subset_percent=coco_subset_percent,
    normalize=True
)

print(f'\nContent images:')
print(f'  Training set: {x_train.shape}')
print(f'  Test set: {x_test.shape}')

### 2.2. Download and load style images

In [ ]:
# Download famous artworks
style_paths = download_style_images(
    data_dir='../data/styles',
    target_size=(image_size, image_size)
)

# Load into memory
style_images, style_metadata = load_style_images(
    data_dir='../data/styles',
    normalize=True
)

print(f'\nLoaded {len(style_images)} style images:')
for key, info in style_metadata.items():
    print(f"  - {info['name']} by {info['artist']}")

### 2.3. Visualize style images

In [ ]:
# Display all style images
n_styles = len(style_images)
n_cols = min(5, n_styles)
n_rows = int(np.ceil(n_styles / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
axes = np.array(axes).reshape(-1)

for idx, (key, img) in enumerate(style_images.items()):
    axes[idx].imshow(img)
    axes[idx].set_title(style_metadata[key]['name'], fontsize=10)
    axes[idx].axis('off')

for idx in range(len(style_images), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Style Images (Famous Artworks)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.4. Visualize sample content images

In [ ]:
# Show some content images
n_samples = 10
indices = np.random.choice(len(x_test), n_samples, replace=False)
samples = x_test[indices]

fig = plot_image_grid(samples, n_cols=5, figsize=(10, 4))
plt.suptitle('Sample Content Images (COCO)', fontsize=14, fontweight='bold')
plt.show()

## 3. Style transfer model

### 3.1. Build model

The AdaIN-based style transfer model consists of:
1. **Encoder**: VGG19 (pretrained on ImageNet) extracts features
2. **AdaIN Layer**: Aligns content features with style statistics
3. **Decoder**: Reconstructs stylized image from aligned features

In [ ]:
# Build style transfer model
encoder, decoder, style_transfer_model = build_style_transfer_model(
    input_shape=(image_size, image_size, 3)
)

print("\nStyle transfer model built successfully!")
print(f"Encoder output shape: {encoder.output.shape}")
print(f"Decoder output shape: {decoder.output.shape}")

## 4. Style transfer demo (no training required!)

The beauty of AdaIN is that it works immediately without training! Let's try it:

In [ ]:
def stylize_image(content_idx, style_key):
    """
    Apply style transfer to a content image.
    
    Args:
        content_idx: Index of content image in x_test
        style_key: Key for style image (e.g., 'starry_night')
    """
    content_img = x_test[content_idx]
    style_img = style_images[style_key]
    
    # Generate
    content_batch = np.expand_dims(content_img, 0)
    style_batch = np.expand_dims(style_img, 0)
    stylized = style_transfer_model.predict(
        [content_batch, style_batch],
        verbose=0
    )[0]
    stylized = np.clip(stylized, 0, 1)
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    axes[0].imshow(content_img)
    axes[0].set_title('Content', fontsize=12)
    axes[0].axis('off')
    
    axes[1].imshow(style_img)
    axes[1].set_title(f"Style: {style_metadata[style_key]['name']}", fontsize=12)
    axes[1].axis('off')
    
    axes[2].imshow(stylized)
    axes[2].set_title('Stylized Result', fontsize=12)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return stylized

print("Available styles:", list(style_images.keys()))
print(f"Content images: 0 to {len(x_test)-1}")

### Try it yourself!

In [ ]:
# Example: Try Van Gogh's Starry Night style
result = stylize_image(content_idx=0, style_key='starry_night')

## 5. Comprehensive results visualization

### 5.1. All styles on multiple content images

In [ ]:
# Select content images
n_content = 3
content_indices = np.random.choice(len(x_test), n_content, replace=False)
test_content = x_test[content_indices]

# Create grid showing all style transfers
style_keys = list(style_images.keys())
n_styles = len(style_keys)

fig, axes = plt.subplots(
    n_content, n_styles + 1,
    figsize=((n_styles + 1) * 2, n_content * 2)
)

for row, content_img in enumerate(test_content):
    # Show original content
    axes[row, 0].imshow(content_img)
    axes[row, 0].set_title('Content' if row == 0 else '')
    axes[row, 0].axis('off')
    
    # Apply each style
    for col, style_key in enumerate(style_keys, start=1):
        style_img = style_images[style_key]
        
        content_batch = np.expand_dims(content_img, 0)
        style_batch = np.expand_dims(style_img, 0)
        
        stylized = style_transfer_model.predict(
            [content_batch, style_batch],
            verbose=0
        )[0]
        stylized = np.clip(stylized, 0, 1)
        
        axes[row, col].imshow(stylized)
        if row == 0:
            axes[row, col].set_title(
                style_metadata[style_key]['artist'].split()[-1],
                fontsize=9
            )
        axes[row, col].axis('off')

plt.suptitle('Style Transfer Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Observations and key takeaways

### What works well:
- **Color Transfer**: Mood and color palette from artwork transfers effectively
- **Texture Patterns**: Brushstroke styles and textures are well-preserved
- **Real-Time**: AdaIN allows fast inference (no optimization needed per image!)
- **Arbitrary Styles**: Can transfer any style to any content without retraining

### Limitations:
- **Content-Style Tradeoff**: Increasing style weight may lose content structure
- **Resolution**: Higher resolution requires more memory
- **Semantic Understanding**: Model doesn't understand object boundaries

### Possible extensions:
1. **Semantic Segmentation Masks**: Apply different styles to different objects
2. **Video Style Transfer**: Add temporal consistency for smooth video
3. **User-Controllable Weights**: Interactive sliders for content/style balance
4. **Style Interpolation**: Blend multiple styles together

### Comparison to training-based methods:
- **Gatys et al. (2015)**: Requires optimization for each image pair (slow)
- **AdaIN (2017)**: Single forward pass (fast), but decoder quality depends on training
- **This Notebook**: Uses pretrained VGG + simple decoder for instant results!